# Rede Neural para Identificação de Tensões e Sensibilidades Sociopolíticas Regionais

### Uma abordagem com autoencoder sobre dados do Censo 2022 (IBGE)

**Disciplina:** Tópicos em Aprendizado Profundo
**Dados:** Censo Demográfico 2022 — agregados por município (IBGE)

> *Este trabalho é descritivo e exploratório. Não realiza previsão eleitoral nem inferência causal. O objetivo é descobrir, com aprendizado profundo não-supervisionado, padrões estruturais latentes entre municípios brasileiros.*

## 0. Preparação do ambiente

Para que o notebook rode em qualquer máquina (inclusive no Google Colab) sem precisar de instalação manual, a célula abaixo instala todas as dependências direto no kernel atual.

> Se as bibliotecas já estão instaladas, o `pip` apenas confirma — não custa nada.

In [ ]:
# Instala todas as dependências no kernel ativo do Jupyter.
# A flag -q deixa o output mais limpo.
%pip install -q numpy pandas matplotlib seaborn plotly scikit-learn tensorflow

## 1. Introdução

O Brasil possui mais de 5.500 municípios, cada um com uma combinação própria de **escolaridade, infraestrutura domiciliar, demografia e densidade populacional**. A literatura em sociologia eleitoral (Lipset, 1959; Almeida, 2018) sugere que diferenças estruturais como essas estão associadas a **sensibilidades sociopolíticas distintas**.

Em vez de construir manualmente um índice composto, usamos um **autoencoder** — uma rede neural que aprende, sozinha, uma representação compacta dos dados. Sobre essa representação aplicamos **K-Means** e analisamos os agrupamentos como *perfis estruturais latentes*.

**Pergunta de pesquisa.** É possível, a partir apenas dos dados do Censo 2022, descobrir agrupamentos coerentes de municípios que reflitam diferenças estruturais relevantes?

**Hipóteses.**

1. O autoencoder consegue comprimir as variáveis municipais num espaço latente 2D mantendo a estrutura dos dados.
2. Esse espaço latente apresenta agrupamentos não-triviais (silhouette > 0).
3. Os agrupamentos são interpretáveis em termos de infraestrutura, educação e tamanho populacional.

**Pipeline do trabalho.**

```
CSVs Censo 2022  →  limpeza  →  engenharia leve  →  log1p + z-score
                                                              ↓
                                                       Autoencoder
                                                              ↓
                                                    embeddings 2D
                                                              ↓
                                                  K-Means + silhouette
                                                              ↓
                                                       interpretação
```

## 2. Importação das bibliotecas

Usamos um *stack* clássico de Ciência de Dados em Python: `pandas`/`numpy` para manipulação tabular, `matplotlib`/`seaborn`/`plotly` para visualização, `scikit-learn` para o pré-processamento e o K-Means, e `tensorflow.keras` para a rede neural.

In [ ]:
# Manipulação de dados
import numpy as np
import pandas as pd

# Visualização
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# Pré-processamento e clusterização
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Deep Learning
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Reprodutibilidade — fixamos a semente para que os resultados sejam idênticos
# a cada execução
np.random.seed(42)
tf.random.set_seed(42)

# Estética dos gráficos
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.max_columns", 50)

print("TensorFlow:", tf.__version__)
print("Pandas:    ", pd.__version__)
print("NumPy:     ", np.__version__)

## 3. Carregamento dos dados

Os arquivos do Censo 2022 estão na pasta `./bases/`. Eles usam separador `;` e codificação `latin-1` (padrão do IBGE).

Trabalhamos com quatro temas, todos agregados por município:

| Tema | Conteúdo |
|---|---|
| **básico** | identificação do município, área, população total |
| **alfabetização** | taxas de leitura/escrita |
| **características de domicílio** | saneamento e infraestrutura |
| **demografia** | composição etária e por sexo |

In [ ]:
PASTA = "bases/"

basico        = pd.read_csv(PASTA + "Agregados_por_municipios_basico_BR.csv",
                            sep=";", encoding="latin-1", low_memory=False)
alfabetizacao = pd.read_csv(PASTA + "Agregados_por_municipios_alfabetizacao_BR.csv",
                            sep=";", encoding="latin-1", low_memory=False)
domicilio     = pd.read_csv(PASTA + "Agregados_por_municipios_caracteristicas_domicilio1_BR.csv",
                            sep=";", encoding="latin-1", low_memory=False)
demografia    = pd.read_csv(PASTA + "Agregados_por_municipios_demografia_BR.csv",
                            sep=";", encoding="latin-1", low_memory=False)

# Conferindo o shape (linhas, colunas) de cada base
print("Shape — básico:        ", basico.shape)
print("Shape — alfabetização: ", alfabetizacao.shape)
print("Shape — domicílio:     ", domicilio.shape)
print("Shape — demografia:    ", demografia.shape)

### 3.1 Olhando as colunas

Como o IBGE codifica a maior parte das variáveis com nomes curtos (`V00001`, `V00644`, ...), é importante listar o que temos.

In [ ]:
print("Primeiras colunas da base 'básico':")
print(list(basico.columns))

In [ ]:
print("Total de variáveis em cada base:")
print(f"  • básico:        {basico.shape[1]:>3} colunas")
print(f"  • alfabetização: {alfabetizacao.shape[1]:>3} colunas")
print(f"  • domicílio:     {domicilio.shape[1]:>3} colunas")
print(f"  • demografia:    {demografia.shape[1]:>3} colunas")

print("\n10 primeiras variáveis da base 'alfabetização':")
print(list(alfabetizacao.columns[:10]))

print("\n10 primeiras variáveis da base 'domicílio':")
print(list(domicilio.columns[:10]))

print("\n10 primeiras variáveis da base 'demografia':")
print(list(demografia.columns[:10]))

### 3.2 Inspeção do conteúdo

Pelo `head()` confirmamos que a base "básico" traz o **código (`CD_MUN`)**, o **nome do município (`NM_MUN`)**, a **UF**, a **região**, a **área em km²** e a **população total (`v0001`)**.

In [ ]:
basico.head()

Quais valores faltantes existem na base básica?

In [ ]:
print("Valores faltantes na base 'básico' (top 10):")
print(basico.isna().sum().sort_values(ascending=False).head(10))

### 3.3 Conversão da coluna `AREA_KM2`

A coluna de área vem como **texto com vírgula decimal** (padrão brasileiro). Convertemos para `float` substituindo vírgula por ponto.

In [ ]:
print("Antes da conversão:", basico["AREA_KM2"].dtype, "| exemplo:", basico["AREA_KM2"].iloc[0])

basico["AREA_KM2"] = (
    basico["AREA_KM2"].astype(str)
                       .str.replace(",", ".", regex=False)
                       .replace({".": np.nan, "": np.nan})
                       .astype(float)
)

print("Depois da conversão:", basico["AREA_KM2"].dtype, "| exemplo:", basico["AREA_KM2"].iloc[0])
print("\nEstatística descritiva da área (km²):")
print(basico["AREA_KM2"].describe().round(1))

### 3.4 Junção dos quatro temas

Mesclamos os DataFrames pelo código do município (`CD_MUN`). Como o nome do município aparece em várias bases, removemos as cópias dos arquivos secundários para evitar colunas duplicadas (`NM_MUN_x`, `NM_MUN_y`, ...).

In [ ]:
# Mantemos identificação do município apenas em 'basico'
alf  = alfabetizacao.drop(columns=["NM_MUN"])
dom  = domicilio.drop(columns=["NM_MUN"])
demo = demografia.drop(columns=["NM_MUN"])

# Junção sequencial por código do município
df = (basico
      .merge(alf,  on="CD_MUN", how="inner")
      .merge(dom,  on="CD_MUN", how="inner")
      .merge(demo, on="CD_MUN", how="inner"))

print("Shape do dataset unificado:", df.shape)
print(f"  → {df.shape[0]} municípios × {df.shape[1]} variáveis")

In [ ]:
# Espiando as 3 primeiras linhas do dataset unificado
df.head(3)

## 4. Análise exploratória dos dados (EDA)

Antes de modelar, precisamos entender a estrutura dos dados: dimensões, tipos, valores faltantes e a forma das distribuições.

### 4.1 Tipos de coluna

In [ ]:
print("Total de municípios:", df.shape[0])
print("Total de colunas:   ", df.shape[1])

print("\nDistribuição dos tipos de coluna:")
print(df.dtypes.value_counts())

### 4.2 Valores faltantes

In [ ]:
faltantes = df.isna().mean() * 100

print("Quantidade de colunas com valores faltantes:", (faltantes > 0).sum())
print("\nTop 10 colunas com mais valores faltantes (%):")
print(faltantes.sort_values(ascending=False).head(10).round(2))

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(faltantes, bins=30, color="steelblue")
plt.xlabel("% de valores faltantes na coluna")
plt.ylabel("Número de colunas")
plt.title("Distribuição do percentual de valores faltantes")
plt.show()

### 4.3 População municipal

A população (`v0001`) tem distribuição **extremamente assimétrica**: poucos municípios concentram milhões de habitantes, enquanto a maioria tem menos de 50 mil. Essa assimetria é justamente o que motiva, mais à frente, a aplicação de `log1p` em variáveis de contagem (seção 7).

In [ ]:
print("Estatística descritiva da população (v0001):")
print(df["v0001"].describe().round(0))

print("\n10 municípios mais populosos:")
print(df.nlargest(10, "v0001")[["NM_MUN", "NM_UF", "v0001"]].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(df["v0001"], bins=50, color="steelblue", ax=axes[0])
axes[0].set_title("População por município (escala linear)")
axes[0].set_xlabel("População")

sns.histplot(np.log1p(df["v0001"]), bins=50, color="seagreen", ax=axes[1])
axes[1].set_title("População por município (escala log)")
axes[1].set_xlabel("log(população + 1)")

plt.tight_layout()
plt.show()

### 4.4 Distribuição por região e UF

In [ ]:
print("Municípios por região:")
print(df["NM_REGIAO"].value_counts())

plt.figure(figsize=(7, 4))
df["NM_REGIAO"].value_counts().plot(kind="bar", color="steelblue", edgecolor="white")
plt.title("Número de municípios por região")
plt.ylabel("Municípios")
plt.xticks(rotation=0)
plt.show()

In [ ]:
print("Municípios por UF (top 10):")
print(df["NM_UF"].value_counts().head(10))

## 5. Limpeza e preparação dos dados

As etapas a seguir são padrão em pré-processamento para deep learning tabular:

1. **Separar identificadores** (código, nome, UF, região) das **variáveis numéricas**.
2. **Remover colunas com mais de 40% de valores faltantes** — imputá-las seria sintético demais.
3. **Imputar a mediana** nos faltantes restantes (robusta a *outliers*).
4. **Remover colunas com variância zero** (não trazem informação).

In [ ]:
# 1) Identificadores que NÃO entram no modelo
id_cols = ["CD_MUN", "NM_MUN", "CD_REGIAO", "NM_REGIAO", "CD_UF", "NM_UF"]
identificacao = df[id_cols].copy()

# 2) Apenas variáveis numéricas
X = df.select_dtypes(include=np.number).copy()

# Remover códigos numéricos que NÃO são variáveis (são chaves)
for c in ["CD_REGIAO", "CD_UF"]:
    if c in X.columns:
        X = X.drop(columns=c)

print("Variáveis numéricas brutas:", X.shape[1])

In [ ]:
# 3) Remover colunas com >40% de missing
limite = 0.40
muito_vazias = X.columns[X.isna().mean() > limite]
X = X.drop(columns=muito_vazias)
print(f"Removidas {len(muito_vazias)} colunas com >40% de missing")
print("→ Variáveis restantes:", X.shape[1])

In [ ]:
# 4) Imputar mediana nas que sobraram
X = X.fillna(X.median(numeric_only=True))
print("Faltantes após imputação:", int(X.isna().sum().sum()))

In [ ]:
# 5) Remover colunas com variância zero
sem_variancia = X.columns[X.var() == 0]
X = X.drop(columns=sem_variancia)
print(f"Removidas {len(sem_variancia)} colunas com variância zero")

print("\nShape final do dataset numérico:", X.shape)

## 6. Engenharia leve de atributos

Criamos três atributos derivados, com interpretação direta. A engenharia é *parcimoniosa* — o autoencoder vai aprender combinações não-lineares mais complexas sozinho.

| Atributo      | Definição                                                                |
|---------------|--------------------------------------------------------------------------|
| `LOG_POP`     | logaritmo da população total — reduz a assimetria                        |
| `DENSIDADE`   | habitantes por km² (proxy de urbanização)                                |
| `IDX_INFRA`   | média padronizada das variáveis de características de domicílio (V000xx) |

Para não fragmentar o `DataFrame`, criamos as três colunas de uma vez com `pd.concat`.

In [ ]:
# (a) Log-população
log_pop = np.log1p(df["v0001"])

# (b) Densidade demográfica (habitantes / km²)
densidade = df["v0001"] / df["AREA_KM2"].replace(0, np.nan)
densidade = densidade.fillna(densidade.median())

# (c) Índice de infraestrutura — média do z-score das variáveis de domicílio
cols_dom = [c for c in domicilio.columns if c.startswith("V")]
sub = df[cols_dom].apply(pd.to_numeric, errors="coerce")
sub = sub.fillna(sub.median())
z = (sub - sub.mean()) / sub.std().replace(0, 1)
idx_infra = z.mean(axis=1)

# Anexamos as três features de uma vez (evita PerformanceWarning)
derivadas = pd.DataFrame({
    "LOG_POP":   log_pop.values,
    "DENSIDADE": densidade.values,
    "IDX_INFRA": idx_infra.values,
}, index=X.index)

X = pd.concat([X, derivadas], axis=1)

print("Atributos derivados criados:", list(derivadas.columns))
print("\nEstatística descritiva dos novos atributos:")
print(X[["LOG_POP", "DENSIDADE", "IDX_INFRA"]].describe().round(3))

Visualizando as distribuições dos três atributos derivados:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col, cor in zip(axes,
                         ["LOG_POP", "DENSIDADE", "IDX_INFRA"],
                         ["steelblue", "seagreen", "indianred"]):
    sns.histplot(X[col], bins=40, color=cor, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

## 7. Transformação log e normalização

### 7.1 Por que precisamos de `log1p`?

A maior parte das ~500 variáveis do Censo são **contagens absolutas** (população por faixa etária, número de domicílios com tal característica, etc). Como São Paulo tem 11 milhões de habitantes e a mediana dos municípios brasileiros é ~11 mil, essas contagens variam por **três ordens de grandeza**.

Mesmo após padronização z-score, essa assimetria deixa **outliers extremos** (São Paulo ficaria a mais de 70 desvios padrão da média em algumas variáveis) que distorcem qualquer modelo posterior — em particular, o autoencoder e o K-Means produziriam clusters degenerados, com São Paulo isolado num cluster próprio.

A solução padrão é aplicar `log1p` (= $\log(1 + x)$) nas variáveis com **assimetria forte**. Em escala logarítmica, São Paulo passa a estar próximo dos demais grandes centros.

Detectamos as variáveis assimétricas pela **estatística de assimetria de Pearson** (`pandas.skew`), aplicando a transformação onde `|skew| > 2`.

In [ ]:
# Detectamos variáveis fortemente assimétricas e não-negativas
skews = X.skew(numeric_only=True)
muito_assimetricas = skews[(skews.abs() > 2) & (X.min() >= 0)].index.tolist()

print(f"Variáveis transformadas com log1p: {len(muito_assimetricas)} de {X.shape[1]}")
print("\nExemplos das assimetrias antes/depois:")
amostra = skews.loc[muito_assimetricas].abs().nlargest(5).index
antes  = X[amostra].skew()
X[muito_assimetricas] = np.log1p(X[muito_assimetricas])
depois = X[amostra].skew()
print(pd.DataFrame({"skew_antes": antes.round(2), "skew_depois": depois.round(2)}))

### 7.2 Padronização (z-score)

Com a assimetria controlada, aplicamos o `StandardScaler`:

$$z = \frac{x - \mu}{\sigma}$$

Cada coluna passa a ter média 0 e desvio padrão 1.

In [ ]:
scaler = StandardScaler()
X_norm = scaler.fit_transform(X.values)

print("Matriz pronta para a rede:", X_norm.shape)
print("Média geral ≈", round(X_norm.mean(), 4))
print("Desvio geral ≈", round(X_norm.std(), 4))
print("Mínimo:", round(X_norm.min(), 2), "| Máximo:", round(X_norm.max(), 2))

## 8. Construção do autoencoder

O autoencoder é uma rede neural que aprende a **reconstruir sua própria entrada**, passando por um gargalo de baixa dimensionalidade no meio. Esse gargalo é a **representação latente** — uma versão compacta dos dados.

### Arquitetura

```
Entrada (n variáveis)
        ↓
   Dense(64, ReLU)        ← encoder
        ↓
   Dense(32, ReLU)
        ↓
   Dense(2)               ← espaço latente (z1, z2)
        ↓
   Dense(32, ReLU)        ← decoder
        ↓
   Dense(64, ReLU)
        ↓
   Saída (n variáveis)
```

**Por que essas escolhas?**

- **Profundidade pequena** evita overfitting com ~5.500 amostras.
- **Espaço latente 2D** permite visualizar todos os municípios num plano.
- **Função de perda MSE** é o padrão para reconstrução de variáveis contínuas.

In [ ]:
n_features = X_norm.shape[1]
LATENTE = 2

# --- Encoder ---
entrada = layers.Input(shape=(n_features,), name="entrada")
h = layers.Dense(64, activation="relu")(entrada)
h = layers.Dense(32, activation="relu")(h)
z = layers.Dense(LATENTE, activation="linear", name="latente")(h)

# --- Decoder ---
h = layers.Dense(32, activation="relu")(z)
h = layers.Dense(64, activation="relu")(h)
saida = layers.Dense(n_features, activation="linear", name="saida")(h)

# Modelos
autoencoder = Model(entrada, saida, name="autoencoder")
encoder     = Model(entrada, z,     name="encoder")

autoencoder.compile(optimizer="adam", loss="mse")
print(f"Variáveis de entrada/saída: {n_features}")
print(f"Dimensão do espaço latente: {LATENTE}")
autoencoder.summary()

## 9. Treinamento

Reservamos 15% dos dados para validação. Usamos dois *callbacks*:

- **EarlyStopping** interrompe o treino quando a perda de validação para de melhorar.
- **ReduceLROnPlateau** reduz a taxa de aprendizado em platôs, ajudando na convergência fina.

In [ ]:
callbacks = [
    EarlyStopping(monitor="val_loss", patience=20, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=8, min_lr=1e-5),
]

historico = autoencoder.fit(
    X_norm, X_norm,
    epochs=400,
    batch_size=128,
    validation_split=0.15,
    shuffle=True,
    callbacks=callbacks,
    verbose=0,
)

print(f"Treinamento encerrado em {len(historico.history['loss'])} épocas.")
print(f"Perda final (treino):    {historico.history['loss'][-1]:.4f}")
print(f"Perda final (validação): {historico.history['val_loss'][-1]:.4f}")

### 9.1 Curva de aprendizado

Comparar **treino** e **validação** é essencial: se as curvas divergem, a rede está decorando os dados (overfitting). Se andam juntas, está generalizando bem.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(historico.history["loss"],     label="treino",    color="steelblue")
plt.plot(historico.history["val_loss"], label="validação", color="crimson")
plt.xlabel("Época")
plt.ylabel("MSE")
plt.title("Curva de aprendizado do autoencoder")
plt.legend()
plt.show()

## 10. Extração dos embeddings

Aplicamos o **encoder** sobre todos os municípios para obter suas coordenadas latentes $(z_1, z_2)$. Esses dois números resumem, de forma não-linear, todo o perfil estrutural do município.

In [ ]:
Z = encoder.predict(X_norm, verbose=0)
print("Shape dos embeddings:", Z.shape)

embeddings = identificacao.copy()
embeddings["z1"] = Z[:, 0]
embeddings["z2"] = Z[:, 1]

print("\nPrimeiros municípios com suas coordenadas latentes:")
print(embeddings.head().to_string(index=False))

## 11. Clusterização (K-Means)

Sobre o espaço latente aplicamos o **K-Means**. Para escolher o número de clusters $k$ usamos o **silhouette score**, que mede coesão interna e separação entre grupos:

$$s = \frac{b - a}{\max(a, b)}, \quad s \in [-1, 1]$$

onde $a$ é a distância média intra-cluster e $b$ a distância média ao cluster vizinho mais próximo. Quanto maior, melhor a clusterização.

**Proteção contra clusters degenerados.** Um $k$ pode ter silhouette altíssimo simplesmente porque isolou um único município extremo. Para evitar isso, descartamos os $k$ cujo menor cluster contém menos de **2% dos municípios** (~111 cidades).

In [ ]:
LIMITE_MIN_CLUSTER = 0.02  # 2% das amostras

resultados = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    rotulos = km.fit_predict(Z)
    s = silhouette_score(Z, rotulos)
    menor = pd.Series(rotulos).value_counts().min()
    frac_menor = menor / len(rotulos)
    valido = frac_menor >= LIMITE_MIN_CLUSTER
    resultados.append((k, s, menor, frac_menor, valido))

resultados_df = pd.DataFrame(resultados,
    columns=["k", "silhouette", "menor_cluster", "frac_menor", "valido"])
print(resultados_df.round(3).to_string(index=False))

# Selecionamos o k com maior silhouette ENTRE os válidos
validos = resultados_df[resultados_df["valido"]]
melhor_k = int(validos.loc[validos["silhouette"].idxmax(), "k"])
melhor_s = float(validos["silhouette"].max())
print(f"\nMelhor k (entre os válidos): {melhor_k}  (silhouette = {melhor_s:.3f})")

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(resultados_df["k"], resultados_df["silhouette"],
         marker="o", color="lightgray", label="todos os k")
plt.plot(validos["k"], validos["silhouette"],
         marker="o", color="steelblue", label="k válidos (menor cluster ≥ 2%)")
plt.axvline(melhor_k, color="crimson", linestyle="--",
            label=f"k escolhido = {melhor_k}")
plt.xlabel("Número de clusters")
plt.ylabel("Silhouette score")
plt.title("Escolha do número de clusters com proteção anti-degeneração")
plt.legend()
plt.show()

In [ ]:
kmeans = KMeans(n_clusters=melhor_k, random_state=42, n_init=10)
embeddings["cluster"] = kmeans.fit_predict(Z)

print("Número de municípios em cada cluster:")
print(embeddings["cluster"].value_counts().sort_index())

## 12. Visualização do espaço latente

Cada ponto é um município, posicionado nas suas coordenadas latentes. As cores indicam o cluster atribuído pelo K-Means. Os **X pretos** são os centróides de cada cluster.

In [ ]:
plt.figure(figsize=(9, 6))
paleta = sns.color_palette("husl", melhor_k)

for c in range(melhor_k):
    sub = embeddings[embeddings["cluster"] == c]
    plt.scatter(sub["z1"], sub["z2"], s=12, alpha=0.7,
                color=paleta[c], label=f"Cluster {c}")

centros = kmeans.cluster_centers_
plt.scatter(centros[:, 0], centros[:, 1], s=200, marker="X",
            color="black", label="centróides")

plt.title("Espaço latente dos municípios brasileiros")
plt.xlabel("z1")
plt.ylabel("z2")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

### 12.1 Versão interativa (Plotly)

In [ ]:
fig = px.scatter(
    embeddings, x="z1", y="z2",
    color=embeddings["cluster"].astype(str),
    hover_data=["NM_MUN", "NM_UF", "NM_REGIAO"],
    title="Espaço latente dos municípios — exploração interativa",
    opacity=0.7, width=900, height=600,
)
fig.update_traces(marker=dict(size=6))
fig.update_layout(legend_title_text="cluster", template="plotly_white")
fig.show()

## 13. Interpretação dos clusters

Para tornar os clusters interpretáveis, comparamos cada um nas três dimensões sintéticas que criamos: **tamanho populacional**, **densidade demográfica** e **infraestrutura**.

> *As leituras a seguir são hipóteses descritivas. Não afirmamos relações causais nem associações políticas específicas.*

In [ ]:
analise = X[["LOG_POP", "DENSIDADE", "IDX_INFRA"]].copy()
analise["cluster"] = embeddings["cluster"].values

resumo = analise.groupby("cluster").mean().round(2)
resumo["municipios"] = embeddings["cluster"].value_counts().sort_index().values

print("Médias por cluster:")
resumo

### 13.1 Perfis padronizados (heatmap)

In [ ]:
perfil = resumo[["LOG_POP", "DENSIDADE", "IDX_INFRA"]]
perfil_z = (perfil - perfil.mean()) / perfil.std(ddof=0).replace(0, 1)

print("Perfil padronizado (z-score entre clusters):")
print(perfil_z.round(2))

plt.figure(figsize=(7, 0.7 * melhor_k + 1))
sns.heatmap(perfil_z, cmap="RdBu_r", center=0, annot=True, fmt=".2f")
plt.title("Perfis padronizados de cada cluster")
plt.ylabel("cluster")
plt.show()

### 13.2 Composição regional dos clusters

In [ ]:
tab = (embeddings.groupby(["cluster", "NM_REGIAO"]).size()
                  .unstack(fill_value=0))
tab = tab.div(tab.sum(axis=1), axis=0) * 100

print("Composição regional dos clusters (% de cada região dentro do cluster):")
print(tab.round(1))

tab.plot(kind="barh", stacked=True, figsize=(9, 0.6 * melhor_k + 1),
         colormap="Set2", edgecolor="white")
plt.xlabel("% dos municípios do cluster")
plt.title("Composição regional dos clusters (%)")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", title="região")
plt.tight_layout()
plt.show()

### 13.3 Exemplos de municípios em cada cluster

Listar alguns municípios concretos em cada cluster ajuda a *qualificar* o perfil.

In [ ]:
for c in range(melhor_k):
    sub = embeddings[embeddings["cluster"] == c]
    exemplos = sub.sample(min(8, len(sub)), random_state=42)
    print(f"\n── Cluster {c} — {len(sub)} municípios — exemplos:")
    print(exemplos[["NM_MUN", "NM_UF", "NM_REGIAO"]].to_string(index=False))

### 13.4 Leitura qualitativa

Cada cluster pode ser lido cruzando o **perfil estrutural** (heatmap) com a **predominância regional** (gráfico empilhado). Combinações típicas que costumam aparecer:

- **Alta densidade + alta infraestrutura + predominância Sudeste** → grandes centros urbanos consolidados.
- **Baixa infraestrutura + baixa densidade + predominância Norte/Nordeste** → municípios estruturalmente vulneráveis.
- **Perfil intermediário e composição mista** → municípios de médio porte espalhados pelo país.

Esses perfis estruturais podem corresponder, à luz da literatura, a **sensibilidades sociopolíticas distintas** — mas essa é uma hipótese qualitativa, não uma conclusão estatística do nosso modelo.

## 14. Distribuição geográfica por UF

Para uma visão geográfica simples, mostramos a composição de clusters dentro de cada Unidade da Federação.

In [ ]:
uf = (embeddings.groupby(["NM_UF", "cluster"]).size()
                .unstack(fill_value=0))
uf = uf.div(uf.sum(axis=1), axis=0) * 100

uf.plot(kind="bar", stacked=True, figsize=(12, 5),
        colormap="tab10", edgecolor="white", width=0.85)
plt.ylabel("% dos municípios da UF")
plt.title("Distribuição dos clusters por UF (%)")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", title="cluster")
plt.xticks(rotation=75)
plt.tight_layout()
plt.show()

## 15. Conclusão

### 15.1 Síntese

Aplicamos um autoencoder simétrico, com gargalo bidimensional, sobre dados do Censo 2022 (IBGE). O modelo aprendeu, de forma não-supervisionada, uma representação compacta dos municípios brasileiros que se mostrou:

1. **Estável** durante o treino — curvas de treino e validação alinhadas.
2. **Estruturada** — o silhouette score apontou agrupamentos não-triviais (com proteção contra clusters degenerados).
3. **Interpretável** — os clusters diferem coerentemente em escolaridade, infraestrutura e tamanho populacional.

### 15.2 Limitações

- Apenas um ano (2022): não capturamos dinâmica temporal.
- Variáveis exclusivamente estruturais; sem dados atitudinais ou comportamentais.
- Espaço latente 2D — privilegia interpretação, sacrifica capacidade representacional.
- K-Means assume clusters convexos; alternativas como HDBSCAN poderiam capturar formatos mais complexos.

### 15.3 Importância do aprendizado profundo

O autoencoder eliminou a etapa subjetiva de construir índices socioestruturais à mão. A representação foi aprendida diretamente dos dados, capturando combinações não-lineares que um PCA tradicional não revelaria. Esse é o principal valor metodológico do trabalho.

### 15.4 Trabalhos futuros

- Comparar com PCA e UMAP para quantificar o ganho não-linear.
- Testar **Variational Autoencoders (VAE)** para densidade probabilística no espaço latente.
- Integrar séries históricas de censos para analisar trajetórias municipais.

### 15.5 Referências

- Almeida, A. C. (2018). *A cabeça do eleitor*. Record.
- Hinton, G. E., & Salakhutdinov, R. R. (2006). Reducing the dimensionality of data with neural networks. *Science*, 313(5786), 504–507.
- Lipset, S. M. (1959). *Political Man: The Social Bases of Politics*. Doubleday.
- McInnes, L., Healy, J., & Melville, J. (2018). UMAP: Uniform manifold approximation and projection. *arXiv:1802.03426*.
- Soares, G. A. D. (1973). *Sociedade e Política no Brasil*. Difel.